# Leksikon antar tingkat, augmentasi Krama Alus, dan merge data latih (3 tingkat)

Jalankan dari folder project (cwd = root repo).

1. **Leksikon** ditambang dari *train split saja*: kalimat dalam satu `group` bermakna sama, jadi kata yang berbeda pada posisi sejajar adalah padanan antar tingkat.
2. **Augmentasi**: group di train yang punya Krama tetapi tidak punya Krama Alus dibuatkan Krama Alus sintetis lewat substitusi kata.
3. **Uji kualitas di valid**: aturan yang sama diterapkan ke Krama di valid, lalu dibandingkan dengan Krama Alus asli.
4. **Merge**: Unggah-Ungguh (inti) + augmentasi + NusaX-MT (Ngoko, opsional) -> `data/merged/train_merged_3lv.csv`.

Valid dan test **tidak diubah**. Output sintetis belum diperiksa penutur Jawa.

In [1]:
import re, difflib, collections
from pathlib import Path
import pandas as pd, sacrebleu

SPLIT = Path("data/unggah_ungguh/split")
NUSAX = Path("data/external/nusax/train.csv")   # opsional; CC BY-SA
OUT_LEX, OUT_MERGED = Path("data/lexicon"), Path("data/merged")
LEVELS = ["ngoko", "krama", "krama_alus"]
MIN_COUNT, MIN_RATIO = 2, 0.6      # syarat pasangan kata dianggap andal
NUSAX_MAX_WORDS = 40               # kalimat lebih panjang akan terpotong max_length 64 token

load = lambda s: pd.read_csv(SPLIT / f"{s}.csv").query("level in @LEVELS").reset_index(drop=True)
train, valid, test = load("train"), load("valid"), load("test")
print({k: len(v) for k, v in [("train", train), ("valid", valid), ("test", test)]})

{'train': 1395, 'valid': 175, 'test': 176}


## 1. Menambang leksikon dari train split

In [2]:
WORD = re.compile(r"[^\W\d_]+(?:[-'][^\W\d_]+)*")
tok = lambda s: WORD.findall(str(s).lower())

def mine(df, a, b):
    """Hitung pasangan kata a->b (berubah) dan kata yang tetap, dari group yang punya kedua tingkat."""
    changed, same, n_group = collections.Counter(), collections.Counter(), 0
    for _, g in df.groupby("group"):
        A, B = g[g["level"] == a]["javanese"].tolist(), g[g["level"] == b]["javanese"].tolist()
        if not A or not B:
            continue
        n_group += 1
        ta, tb = tok(A[0]), tok(B[0])
        for op, i1, i2, j1, j2 in difflib.SequenceMatcher(a=ta, b=tb, autojunk=False).get_opcodes():
            if op == "equal":
                same.update(ta[i1:i2])
            elif op == "replace" and i2 - i1 == j2 - j1:      # hanya blok sejajar 1:1
                changed.update(zip(ta[i1:i2], tb[j1:j2]))
    total, best = collections.Counter(), {}
    for (u, v), n in changed.items():
        total[u] += n
        if n > best.get(u, ("", 0))[1]:
            best[u] = (v, n)
    rows = []
    for (u, v), n in changed.items():
        ratio = n / (total[u] + same[u])
        rows.append({"src": u, "tgt": v, "count": n, "src_unchanged": same[u], "tgt_unchanged": same[v],
                     "ratio": round(ratio, 3),
                     # tgt_unchanged >= count: target adalah kata umum (mis. "ingkang"), tanda salah sejajar
                     "reliable": n >= MIN_COUNT and ratio >= MIN_RATIO and best[u][0] == v and same[v] < n})
    lex = pd.DataFrame(rows).sort_values(["reliable", "count"], ascending=False).reset_index(drop=True)
    print(f"{a} -> {b}: {n_group} group, {len(lex)} pasangan unik, {int(lex.reliable.sum())} andal")
    return lex

lex_ka = mine(train, "krama", "krama_alus")
lex_nk = mine(train, "ngoko", "krama")
OUT_LEX.mkdir(parents=True, exist_ok=True)
lex_ka.to_csv(OUT_LEX / "krama_to_krama_alus.csv", index=False, encoding="utf-8")
lex_nk.to_csv(OUT_LEX / "ngoko_to_krama.csv", index=False, encoding="utf-8")
lex_ka[lex_ka.reliable].head(25)

krama -> krama_alus: 206 group, 286 pasangan unik, 57 andal


ngoko -> krama: 592 group, 847 pasangan unik, 242 andal


,src,tgt,count,src_unchanged,tgt_unchanged,ratio,reliable
0,sampeyan,panjenengan,57,0,0,0.679,True
1,kula,adalem,38,2,0,0.950,True
2,gadhah,kagungan,14,2,0,0.875,True
3,anakipun,putranipun,10,0,0,1.000,True
4,piyambakipun,panjenenganipun,10,1,0,0.909,True
5,mangga,sumangga,10,0,0,1.000,True
6,lenggah,pinarak,9,0,0,1.000,True
7,kesah,tindak,9,0,0,0.900,True
8,ngangge,ngagem,8,0,1,1.000,True
9,estri,putri,7,0,1,1.000,True


`ratio` = seberapa sering kata sumber benar-benar diganti dengan kata target saat keduanya sejajar. Kata yang sering tetap sama di Krama Alus tidak lolos; ini menyaring substitusi yang tergantung konteks. `tgt_unchanged` menyaring target yang sebenarnya kata umum (mis. *anak -> ingkang* dari "anak sampeyan" -> "ingkang putra").

## 2. Fungsi substitusi Krama -> Krama Alus

In [3]:
SUB = dict(lex_ka[lex_ka.reliable][["src", "tgt"]].values)

def to_krama_alus(sentence):
    """Ganti kata Krama dengan padanan Krama Alus dari leksikon. Kembalikan (kalimat baru, jumlah substitusi)."""
    n = 0
    def rep(m):
        nonlocal n
        w = m.group(0)
        t = SUB.get(w.lower())
        if t is None:
            return w
        n += 1
        return t.capitalize() if w[0].isupper() else t
    return WORD.sub(rep, str(sentence)), n

for s in ["Sampeyan gadhah griya enggal.", "Bapak sampun mantuk saking kantor."]:
    print(s, "->", to_krama_alus(s))

Sampeyan gadhah griya enggal. -> ('Panjenengan kagungan dalem enggal.', 3)
Bapak sampun mantuk saking kantor. -> ('Bapak sampun kondur saking kantor.', 1)


## 3. Uji kualitas di valid (referensi Krama Alus asli)

In [4]:
def check(df, name):
    src, ref, hyp, nsub = [], [], [], 0
    for _, g in df.groupby("group"):
        K, KA = g[g["level"] == "krama"]["javanese"].tolist(), g[g["level"] == "krama_alus"]["javanese"].tolist()
        if K and KA:
            h, n = to_krama_alus(K[0])
            src.append(K[0]); ref.append(KA[0]); hyp.append(h); nsub += n
    chrf = lambda H: sacrebleu.corpus_chrf(H, [ref], word_order=2).score
    exact = lambda H: sum(a.strip() == b.strip() for a, b in zip(H, ref))
    changed = sum(a != b for a, b in zip(src, hyp))
    print(f"[{name}] {len(ref)} pasangan Krama/Krama Alus | kalimat yang berubah: {changed} | substitusi: {nsub}")
    print(f"  salin Krama apa adanya : chrF++ {chrf(src):.2f} | sama persis {exact(src)}/{len(ref)}")
    print(f"  setelah substitusi     : chrF++ {chrf(hyp):.2f} | sama persis {exact(hyp)}/{len(ref)}")
    return pd.DataFrame({"krama": src, "substitusi": hyp, "krama_alus_asli": ref})

ex = check(valid, "valid")
ex[ex.krama != ex.substitusi].head(10)

[valid] 26 pasangan Krama/Krama Alus | kalimat yang berubah: 22 | substitusi: 43
  salin Krama apa adanya : chrF++ 55.24 | sama persis 0/26
  setelah substitusi     : chrF++ 70.66 | sama persis 1/26


,krama,substitusi,krama_alus_asli
0,"Embah estri mlampah-mlampah dhawah, dipunadega...","Eyang putri mlampah-mlampah dhawah, dipunadega...","Eyang putri tindak-tindak dhawah, dipunjumenen..."
2,"Dipuntumpakanan mesthi sakeca, amargi mobil ku...","Dipuntumpakanan mesthi sakeca, amargi mobil ad...","Dipuntitihana mesti sakeca, amargi mobil adale..."
3,"Kula gadhah pelem, menapa sampeyan ajeng?","Adalem kagungan pelem, menapa panjenengan ajeng?","Kula gadhah pelem, menapa panjenengan kersa?"
4,"Piyambakipun menika ajeng nedha, mila badanipu...","Panjenenganipun menika ajeng dhahar, mila sali...","Panjenenganipun menika kersa dhahar, mila sali..."
5,"Kula namung saged nyukani jam menika, sampeyan...","Adalem namung saged nyukani jam menika, panjen...","Adalem namung saged nyaosi jam menika, panjene..."
6,Griyanipun Pak Purba ingkang pinggir margi sam...,Griyanipun Pak Purba ingkang pinggir margi sam...,Dalemipun Pak Purba ingkang pinggir margi samp...
7,"Wonten ing adegan menika, cariyosipun sampeyan...","Wonten ing adegan menika, cariyosipun panjenen...","Wonten ing adegan mennika, cariyosipun panjene..."
8,Sampeyan criyos piyambak dhateng bapak kula!,Panjenengan criyos piyambak dhateng bapak adalem!,Panjenengan ngandika piyambak dhateng bapak ad...
9,Bapak sampeyan wau mlampah-mlampah kaliyan bap...,Bapak panjenengan wau mlampah-mlampah kaliyan ...,Rama panjenengan (ingkang rama) wau tindak-tin...
10,Mbok (Bok) sampeyan menika badhe kesah dhateng...,Mbok (Bok) panjenengan menika badhe tindak dha...,Ibu panjenengan (ingkang ibu) menika badhe tin...


Angka valid adalah ukuran jujur: leksikon hanya dari train, jadi kalimat valid belum pernah dilihat. Kalau chrF++ setelah substitusi tidak lebih tinggi dari "salin apa adanya", augmentasi ini tidak layak dipakai.

## 4. Augmentasi train: Krama Alus sintetis untuk group tanpa Krama Alus

In [5]:
rows = []
for gid, g in train.groupby("group"):
    if (g["level"] == "krama_alus").any() or not (g["level"] == "krama").any():
        continue
    k = g[g["level"] == "krama"].iloc[0]
    new, n = to_krama_alus(k["javanese"])
    if n > 0:
        rows.append({"group": gid, "level": "krama_alus", "indonesian": k["indonesian"], "javanese": new,
                     "source": "aug_lexicon", "n_sub": n, "krama_asal": k["javanese"]})
aug = pd.DataFrame(rows)
print(f"Krama Alus asli di train: {(train.level == 'krama_alus').sum()} | sintetis baru: {len(aug)}")
aug[["indonesian", "krama_asal", "javanese", "n_sub"]].head(12)

Krama Alus asli di train: 210 | sintetis baru: 213


,indonesian,krama_asal,javanese,n_sub
0,"Jika merah-merahan dengan kepunyaanmu, masih m...","Yen abrit-abritan kaliyan gadhahan sampeyan, t...",Yen abrit-abritan kaliyan gadhahan panjenengan...,2
1,"Ini, lipstikmu memerahi baju saya!","Menika, lipstik sampeyan ngabrit-abriti rasuka...","Menika, lipstik panjenengan ngabrit-abriti ras...",2
2,Jika hanya membiayai sekolah saja saya tidak k...,Yen namung ngragadi sekolah kemawon kula boten...,Yen namung ngragadi sekolah kemawon adalem bot...,1
3,Berat-berat membawa ketela itu akan dipergunak...,Awrat-awrat mbekta tela menika badhe dipunangg...,Awrat-awrat ngampil tela menika badhe dipunage...,2
4,"Jika menanggung biaya kuliah juga, apa tidak m...","Yen nanggel ragad kuliah menapa, menapa boten ...","Yen nanggel ragad kuliah menapa, menapa boten ...",1
5,Saya merasa terlalu jauh jika berlari hingga d...,Kula rumaos ketebihen yen mlajeng ngantos kali...,Adalem rumaos ketebihen yen mlajeng ngantos ka...,1
6,Mari jauh-jauhan melemparkan bola kasti ini!,Mangga tebih-tebihan nyawataken bal kasti menika!,Sumangga tebih-tebihan nyawataken bal kasti me...,1
7,Lemparanmu jauh-jauhnya kira-kira hanya duapul...,Enggen sampeyan nyawataken tebih-tebihipun nam...,Enggen panjenengan nyawataken tebih-tebihipun ...,1
8,"Biarpun kamu memandang sejauh mungkin, yang ad...","Sanajan sampeyan ningali satebih-tebihipun, wo...",Sanajan panjenengan mriksani satebih-tebihipun...,2
9,"Dia hanya mengadu, supaya kamu benci kepada Bu...","Piyambakipun menika namung aben-aben, supados ...","Panjenenganipun menika namung aben-aben, supad...",2


## 5. NusaX-MT (opsional, Ngoko) dan merge

In [6]:
COLS = ["group", "level", "indonesian", "javanese", "source"]
parts = [train.assign(source="unggah_ungguh")[COLS], aug[COLS]]
held = set(valid.indonesian.str.lower()) | set(test.indonesian.str.lower())

if NUSAX.exists():
    nx = pd.read_csv(NUSAX)[["indonesian", "javanese"]].dropna()
    nx = nx[nx.javanese.str.split().str.len() <= NUSAX_MAX_WORDS]
    nx = nx[~nx.indonesian.str.lower().isin(held)]
    # cek tingkat secara kasar: kata khas Krama (target leksikon ngoko->krama yang andal) di kalimat NusaX
    krama_words = set(lex_nk[lex_nk.reliable].tgt) - set(lex_nk.src)
    share = nx.javanese.map(lambda s: sum(w in krama_words for w in tok(s)) / max(1, len(tok(s))))
    print(f"NusaX: {len(nx)} kalimat | rata-rata porsi kata khas Krama: {share.mean():.3f} | kalimat dengan >10% kata Krama: {(share > 0.10).sum()}")
    nx = nx[share <= 0.10].assign(level="ngoko", source="nusax", group=-1)
    parts.append(nx[COLS])
else:
    print("NusaX tidak ditemukan, dilewati:", NUSAX)

merged = pd.concat(parts, ignore_index=True)
assert not merged.indonesian.str.lower().isin(held).any(), "bocor ke valid/test"
OUT_MERGED.mkdir(parents=True, exist_ok=True)
merged.to_csv(OUT_MERGED / "train_merged_3lv.csv", index=False, encoding="utf-8")
print("tersimpan:", OUT_MERGED / "train_merged_3lv.csv", len(merged), "baris")
pd.crosstab(merged.source, merged.level, margins=True)

NusaX: 421 kalimat | rata-rata porsi kata khas Krama: 0.004 | kalimat dengan >10% kata Krama: 1
tersimpan: data\merged\train_merged_3lv.csv 2028 baris


level,krama,krama_alus,ngoko,All
source,,,,
aug_lexicon,0,213,0,213
nusax,0,0,420,420
unggah_ungguh,592,210,593,1395
All,592,423,1013,2028


## Catatan pemakaian
- Kolom `source` memungkinkan ablation: latih dengan/tanpa `aug_lexicon` dan `nusax`, uji di 176 kalimat test yang sama.
- NusaX memakai gaya Ngoko yang cenderung Jawa Timur dan domain ulasan; label `ngoko`-nya berasal dari pemeriksaan kasar, bukan anotasi.
- Lisensi: Unggah-Ungguh CC BY-NC 4.0, NusaX CC BY-SA. Cek dulu sebelum file gabungan dipublikasikan.
- Kalimat sintetis belum diperiksa penutur Jawa; substitusi kata per kata bisa keliru saat kata hormat tidak pantas dipakai (mis. untuk diri sendiri).